# Offline Ternary Hex Animation from Saved Runs

This notebook loads saved run artifacts and builds ternary hex-density animations without rerunning training.

Expected run payloads:
- dict with `data` containing snapshot list, or
- raw snapshot list

Snapshot format expected from `SelectionMethod.helper`: each snapshot has keys like `yh` / `yvh` (log-probs).

In [ ]:
from pathlib import Path
import math
import numpy as np
import torch
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.animation import FuncAnimation, FFMpegWriter
from matplotlib.patches import Polygon
from IPython.display import HTML

In [ ]:
def load_run_payload(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    return torch.load(path, map_location='cpu', weights_only=False)

def get_ts(bs=320, epochs=200, save_init=5, save_freq=4, len_data=50000):
    t = 0
    ts = [t]
    n_batches = len_data // bs
    for epoch in range(epochs):
        for i in range(n_batches):
            t += 1
            if epoch < save_init and i % (n_batches // save_freq) == 0:
                ts.append(t)

        if save_init <= epoch <= 25:
            ts.append(t)
        elif 25 < epoch <= 65 and epoch % 4 == 0:
            ts.append(t)
        elif epoch > 65 and epoch % 15 == 0 or (epoch == epochs - 1):
            ts.append(t)
    return ts

def extract_snapshots(payload):
    if isinstance(payload, dict) and 'data' in payload:
        snapshots = payload['data']
    else:
        snapshots = payload
    if not isinstance(snapshots, (list, tuple)):
        raise ValueError('Could not find snapshot list in run artifact.')
    if len(snapshots) == 0:
        raise ValueError('Snapshot list is empty.')
    return list(snapshots)


def _to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def snapshot_to_ternary(snapshot, split_key='yh', ternary_mode='first3'):
    if split_key not in snapshot:
        raise KeyError(f"Snapshot missing key '{split_key}'. Available: {list(snapshot.keys())}")

    log_probs = _to_numpy(snapshot[split_key])
    probs = np.exp(log_probs)

    if probs.ndim != 2:
        raise ValueError(f'Expected 2D probabilities, got shape {probs.shape}')

    c = probs.shape[1]
    if c >= 3:
        if ternary_mode == 'first3':
            tri = probs[:, :3]
        elif ternary_mode == 'top3':
            top3_idx = np.argsort(probs, axis=1)[:, -3:]
            tri = np.take_along_axis(probs, top3_idx, axis=1)
        else:
            raise ValueError('ternary_mode must be first3 or top3')
    elif c == 2:
        third = np.clip(1.0 - probs.sum(axis=1, keepdims=True), 0.0, 1.0)
        tri = np.concatenate([probs, third], axis=1)
    else:
        raise ValueError(f'Need >=2 classes to project to ternary, got {c}')

    denom = np.clip(tri.sum(axis=1, keepdims=True), 1e-12, None)
    return tri / denom

def barycentric_to_cartesian(bary):
    v0 = np.array([0.0, 0.0])
    v1 = np.array([1.0, 0.0])
    v2 = np.array([0.5, math.sqrt(3) / 2.0])
    return bary[:,0:1]*v0 + bary[:,1:2]*v1 + bary[:,2:3]*v2


def infer_dataset_name_from_run_path(run_path):
    rp = Path(run_path)
    if 'results' not in rp.parts:
        raise ValueError(f'Cannot infer dataset from path: {run_path}')
    idx = rp.parts.index('results')
    if idx + 1 >= len(rp.parts):
        raise ValueError(f'Cannot infer dataset from path: {run_path}')
    return rp.parts[idx + 1]


def load_true_labels_for_run(run_path, split_key='yh'):
    dataset = infer_dataset_name_from_run_path(run_path)
    labels_path = Path('/home/lgreen/projects/Online_BS/results/data') / f'labels_{dataset}.p'
    if not labels_path.exists():
        raise FileNotFoundError(f'Missing label file: {labels_path}')

    labels_payload = torch.load(labels_path, map_location='cpu', weights_only=False)
    if not isinstance(labels_payload, dict):
        raise ValueError(f'Unexpected label payload format in {labels_path}')

    split = 'train' if split_key == 'yh' else 'val'
    if split not in labels_payload:
        raise KeyError(f"Label file missing split '{split}'. Available: {list(labels_payload.keys())}")

    labels = np.asarray(labels_payload[split]).astype(int)
    return labels, labels_path

In [ ]:
def make_ternary_hex_animation(
    snapshots,
    ts,
    title,
    split_key='yh',
    ternary_mode='first3',
    class_index=None,
    true_labels=None,
    gridsize=24,
    cmap='turbo',
    norm='sqrt',
    interval=150,
    corner_labels=('Class 0', 'Class 1', 'Class 2'),
    save_mp4=False,
    out_name='ternary_hex_offline'
):
    y_max = math.sqrt(3) / 2.0

    # Convert all snapshots once so normalization can be fixed globally.
    bary_frames = [snapshot_to_ternary(s, split_key=split_key, ternary_mode=ternary_mode) for s in snapshots]

    # Optional class filtering by TRUE labels (not argmax).
    if class_index is not None:
        cls = int(class_index)
        if cls < 0 or cls > 2:
            raise ValueError('class_index must be one of {0,1,2} or None')
        if true_labels is None:
            raise ValueError('true_labels is required when class_index is set')

        labels_arr = np.asarray(true_labels).astype(int)
        filtered = []
        class_counts = []
        for b in bary_frames:
            if labels_arr.shape[0] != b.shape[0]:
                raise ValueError(
                    f'true_labels length ({labels_arr.shape[0]}) does not match snapshot rows ({b.shape[0]})'
                )
            keep = labels_arr == cls
            bf = b[keep]
            class_counts.append(int(bf.shape[0]))
            if bf.shape[0] == 0:
                # Keep one dummy point so hexbin remains well-defined for empty frames.
                bf = np.array([[1/3, 1/3, 1/3]], dtype=float)
            filtered.append(bf)
        bary_frames = filtered
        vmax = float(max(class_counts) if len(class_counts) > 0 else 1)
    else:
        # Full animation scale is bounded by total points in a frame.
        frame_sizes = [int(b.shape[0]) for b in bary_frames]
        vmax = float(max(frame_sizes) if len(frame_sizes) > 0 else 1)

    vmax = max(vmax, 1.0)
    cart_frames = [barycentric_to_cartesian(b) for b in bary_frames]

    norm_name = str(norm).lower()
    if norm_name == 'linear':
        density_norm = mcolors.Normalize(vmin=0.0, vmax=vmax, clip=True)
    elif norm_name == 'log':
        density_norm = mcolors.LogNorm(vmin=1e-3, vmax=max(vmax, 1e-3 + 1e-8), clip=True)
    else:
        density_norm = mcolors.PowerNorm(gamma=0.5, vmin=0.0, vmax=vmax, clip=True)

    fig, ax = plt.subplots(figsize=(6, 6))
    verts = np.array([[0.0, 0.0], [1.0, 0.0], [0.5, y_max], [0.0, 0.0]])
    ax.plot(verts[:, 0], verts[:, 1], color='black', linewidth=1.2, zorder=5)

    left_label, right_label, top_label = corner_labels
    ax.text(0.0, -0.025, left_label, ha='left', va='top', fontsize=10)
    ax.text(1.0, -0.025, right_label, ha='right', va='top', fontsize=10)
    ax.text(0.5, y_max + 0.006, top_label, ha='center', va='bottom', fontsize=10)

    # Decision boundaries (midpoints to centroid).
    mid_bary = np.array([[0.0, 0.5, 0.5], [0.5, 0.0, 0.5], [0.5, 0.5, 0.0]])
    centroid_bary = np.array([[1/3, 1/3, 1/3]])
    mid_cart = barycentric_to_cartesian(mid_bary)
    centroid_cart = barycentric_to_cartesian(centroid_bary)[0]
    for m in mid_cart:
        ax.plot([m[0], centroid_cart[0]], [m[1], centroid_cart[1]], color='white', linewidth=1.2, zorder=7)

    clip_poly = Polygon(verts[:-1], closed=True, transform=ax.transData)

    hb = ax.hexbin(
        cart_frames[0][:, 0],
        cart_frames[0][:, 1],
        gridsize=gridsize,
        cmap=cmap,
        norm=density_norm,
        mincnt=0,
        extent=(0.0, 1.0, 0.0, y_max),
        linewidths=0.0,
        alpha=0.95,
        zorder=2,
    )
    hb.set_clip_path(clip_poly)
    cbar = fig.colorbar(hb, ax=ax, label='Point density')

    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_xlim(-0.05, 1.05)
    ax.set_ylim(-0.05, y_max + 0.05)

    def update(i):
        nonlocal hb
        hb.remove()
        cur = cart_frames[i]
        hb = ax.hexbin(
            cur[:, 0],
            cur[:, 1],
            gridsize=gridsize,
            cmap=cmap,
            norm=density_norm,
            mincnt=0,
            extent=(0.0, 1.0, 0.0, y_max),
            linewidths=0.0,
            alpha=0.95,
            zorder=2,
        )
        hb.set_clip_path(clip_poly)
        cbar.update_normal(hb)
        cls_txt = 'all' if class_index is None else str(class_index)
        ax.set_title(f'{title} ({corner_labels[class_index] if class_index is not None else cls_txt}) - step {ts[i]}', fontsize=14, pad=22)
        return (hb,)

    ani = FuncAnimation(fig, update, frames=len(cart_frames), interval=interval, blit=False)

    if save_mp4:
        out_path = Path(out_name).with_suffix('.mp4')
        out_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            import imageio_ffmpeg as iioff
            plt.rcParams['animation.ffmpeg_path'] = iioff.get_ffmpeg_exe()
        except Exception:
            pass
        writer = FFMpegWriter(fps=max(1, int(1000 / max(interval, 1))), bitrate=1800)
        ani.save(str(out_path), writer=writer, dpi=150)
        print('Saved', out_path.resolve())

    return fig, ani

In [ ]:
# Find candidate run files
results_root = Path('results')
candidates = sorted(results_root.rglob('*.p')) + sorted(results_root.rglob('*.pt')) + sorted(results_root.rglob('*.pth'))
for p in candidates[:40]:
    print(p)
print(f'Found {len(candidates)} candidate files')

In [ ]:
dataset = 'MakeBlobs'

for method in ["Bayesian", "Full", "RhoLoss", "DivBS", "GradNorm", "Uniform"]:

    # run_path = '/home/lgreen/projects/Online_BS/results/CIFAR3/{"bsel":"' + str(method) + '","seed":16,"model":"Small_cnn","opt":"adamw","bs":320,"ratio":0.1,"lr":0.001,"wd":0.01}.p'
    # run_path = '/home/lgreen/projects/Online_BS/results/'+ str(dataset) + '/{"bsel":"' + str(method) + '","seed":16,"model":"ResNet_torchvision","opt":"adamw","bs":320,"ratio":0.1,"lr":0.001,"wd":0.01}.p'
    run_path = '/home/lgreen/projects/Online_BS/results/MakeBlobs/{"bsel":"' + str(method) + '","seed":16,"model":"Linear","opt":"sgd","bs":50,"ratio":0.1,"lr":0.1,"wd":0.0}.p'

    ts = get_ts(bs=50, epochs=25, save_init=5, save_freq=4, len_data=1000)
    # ts = get_ts(epochs=300, len_data=15000)
    split_key = 'yh'  # 'yh' for train snapshots, 'yvh' for validation snapshots

    if run_path is None:
        print('Set run_path first')
    else:
        payload = load_run_payload(run_path)
        snapshots = extract_snapshots(payload)
        print('Loaded snapshots:', len(snapshots))

        true_labels = None
        labels_path = None
        try:
            true_labels, labels_path = load_true_labels_for_run(run_path, split_key=split_key)
            print('Loaded true labels from:', labels_path)
            print('True label count:', len(true_labels))
        except Exception as e:
            print('Could not load true labels:', e)
            print('If class_index is set, a matching labels_{dataset}.p file is required.')

    # Build and display animation
    if run_path is not None:
        fig, ani = make_ternary_hex_animation(
            snapshots=snapshots,
            ts=ts,
            title=f'{dataset} {method}',
            split_key=split_key,  # keep this aligned with loaded true_labels split
            ternary_mode='first3',
            class_index=0,        # None for all points, or 0/1/2 for one ternary class
            true_labels=true_labels,
            gridsize=48,
            cmap='turbo',
            norm='sqrt',         # 'linear' | 'sqrt' | 'log'
            interval=200,
            # corner_labels=('dog', 'cat', 'bird'),
            save_mp4=True,
            out_name=f'/home/lgreen/projects/Online_BS/runs/hex_frames/{dataset}/0_{method}'
        )
        HTML(ani.to_jshtml())